# SQD for ATP

## Setup

In [ ]:
import matplotlib.pyplot as plt; plt.rcParams.update({"font.family": "serif"})
import numpy as np
import pickle

import pyscf.tools
from pyscf import ao2mo
import pyscf.cc
import pyscf.ao2mo

import qiskit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit import qasm2, qasm3
from qiskit_aer import AerSimulator
import qiskit_ibm_runtime
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.visualization import timeline_drawer
from qiskit.visualization.timeline.stylesheet import IQXStandard, IQXSimple, IQXDebugging

import ffsim

from layout import get_zigzag_physical_layout

In [ ]:
use_initial_measurement: bool = True
use_dd: bool = True

In [ ]:
ibm_computer: str = "ibm_boston"
service = qiskit_ibm_runtime.QiskitRuntimeService(name="Q4BIOFLEX")
computer = service.backend(ibm_computer)
sampler = Sampler(computer)

## Read in the Hamiltonian and ansatz circuit

In [ ]:
fragment = "atp_0_be2_f18"

adapt_iterations: int = 1

In [ ]:
hamiltonian_dir = "hamiltonians"
fcidump = pyscf.tools.fcidump.read(f"{hamiltonian_dir}/{fragment}.fcidump")

norb = fcidump.get("NORB")
num_electrons = fcidump.get("NELEC")
ms2 = fcidump.get("MS2", 0)
ecore = fcidump.get("ECORE")
h1e = fcidump.get("H1")
h2e = fcidump.get("H2")
h2e = pyscf.ao2mo.restore(1, h2e, norb)

n_alpha = (num_electrons + ms2) // 2
n_beta  = (num_electrons - ms2) // 2
nelec = (n_alpha, n_beta)

In [ ]:
mol = pyscf.gto.M()
mol.nelectron = num_electrons
mol.spin = ms2

mf = pyscf.scf.RHF(mol)
mf.get_hcore = lambda *args: h1e
mf.get_ovlp  = lambda *args: np.eye(norb)
mf._eri = pyscf.ao2mo.restore(8, h2e, norb)
mf.kernel()

ccsd = pyscf.cc.RCCSD(mf)
ccsd.kernel()
t1, t2 = ccsd.t1, ccsd.t2

In [ ]:
mol_data = ffsim.MolecularData(
    norb=norb,
    nelec=nelec,
    core_energy=ecore,
    one_body_integrals=h1e,
    two_body_integrals=h2e,
    hf_energy=mf.e_tot,
    ccsd_energy=ccsd.e_tot,
    ccsd_t1=t1,
    ccsd_t2=t2,
)

In [ ]:
# pairs_aa = [(p, p + 1) for p in range(norb - 1)]
# pairs_ab = [(p, p)     for p in range(norb)]
# pairs_aa = [(p, p + 1) for p in range(norb - 1)]
# pairs_ab = [(p, p)     for p in range(0, norb, 4)]
pairs_aa = [(p, p + 1) for p in range(norb - 1)]
pairs_ab = [(p, p)     for p in range(0, norb, 4)]
n_reps = 1

lucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=mol_data.ccsd_t2,
    t1=mol_data.ccsd_t1,
    n_reps=n_reps,
    interaction_pairs=(pairs_aa, pairs_ab),
    optimize=True,
    options=dict(maxiter=50),
    regularization=1e-2,
)

In [ ]:
qubits = qiskit.QuantumRegister(2 * norb, name="q")
circuit = qiskit.QuantumCircuit(qubits)
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), qubits)
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(lucj_op), qubits)

In [ ]:
base = circuit.copy()
base = qiskit.transpiler.passes.RemoveBarriers()(base)
base.count_ops()

In [ ]:
to_run = base.copy()

# Add initial measurement to post-select on all 0.
if use_initial_measurement:
    measure = to_run.copy_empty_like()
    measure.measure_all()
    measure.barrier()
    to_run = measure.compose(to_run)

# Terminal measurement.
to_run.measure_all()
to_run = [to_run]

In [ ]:
to_run[0].count_ops()

In [ ]:
initial_layout, _ = get_zigzag_physical_layout(norb, backend=computer)

In [ ]:
len(initial_layout)

In [ ]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=computer, initial_layout=initial_layout
)
pass_manager.pre_init = ffsim.qiskit.PRE_INIT
to_run = pass_manager.run(to_run)
to_run[0].count_ops()

In [ ]:
len(to_run[0].qubits)

## Prepare to run on hardware

In [ ]:
for c in to_run:
    print(c.count_ops())
    print(sum(v for k, v in c.count_ops().items() if k not in ("measure", "barrier")))

In [ ]:
to_run[0].depth()

In [ ]:
ops = list(to_run[0].count_ops().keys())
num = list(to_run[0].count_ops().values())

In [ ]:
ops = list(to_run[0].count_ops().keys())
num = list(to_run[0].count_ops().values())

plt.bar(ops, num, color="green", edgecolor="black")
plt.xlabel("Gate")
plt.ylabel("Count");

In [ ]:
# to_run[0].draw(fold=-1)

In [ ]:
fig, ax = plt.subplots(figsize=(30, 20));

timeline_drawer(to_run[0], style=IQXSimple(), idle_wires=False, target=computer.target, axis=ax)

### Add DD

In [ ]:
to_run[0].depth()

In [ ]:
if use_dd:
    target = computer.target

    from qiskit.circuit.library import XGate, YGate

    X = XGate()
    Y = YGate()

    dd_sequence = [X, Y, X, Y] * (to_run[0].depth() // 1500)

    from qiskit.transpiler import InstructionProperties

    y_gate_properties = {}
    for qubit in range(target.num_qubits):
        y_gate_properties.update(
            {
                (qubit,): InstructionProperties(
                    duration=target["x"][(qubit,)].duration,
                    error=target["x"][(qubit,)].error,
                )
            }
        )

    target.add_instruction(YGate(), y_gate_properties)

    from qiskit.transpiler import PassManager
    from qiskit.transpiler.passes.scheduling import (
        ALAPScheduleAnalysis,
        PadDynamicalDecoupling,
    )

    dd_pm = PassManager(
        [
            ALAPScheduleAnalysis(target=target),
            PadDynamicalDecoupling(target=target, dd_sequence=dd_sequence),
        ]
    )
    to_run_dd = dd_pm.run(to_run[0])

    # Hacky fix to compile Y gates to native gates.
    computer = service.backend(ibm_computer)
    sampler = Sampler(computer)

    from qiskit.circuit.equivalence_library import (
        SessionEquivalenceLibrary as sel,
    )
    from qiskit.transpiler.passes import BasisTranslator

    to_run_dd = BasisTranslator(sel, list(computer.target.operation_names))(to_run_dd)

    to_run_dd = [to_run_dd]

    labels = ["Raw", "DD (XYXY)"]

    circuits = [to_run[0], to_run_dd[0]]

    all_ops = sorted(set().union(*(c.count_ops().keys() for c in circuits)))

    x = np.arange(len(all_ops))
    width = 0.4
    for i, c in enumerate(circuits):
        counts = c.count_ops()
        aligned_counts = [counts.get(op, 0) for op in all_ops]
        
        x_shifted = x + (i - 0.5) * width
        plt.bar(x_shifted, aligned_counts, width=width,
                edgecolor="black", label=labels[i])

    plt.xticks(x, all_ops)
    plt.xlabel("Gate")
    plt.ylabel("Count")
    plt.legend();
    to_run = to_run_dd

In [ ]:
for c in to_run:
    print(c.count_ops())
    print(sum(c.count_ops().values()))
    print(sum({k: v for k, v in c.count_ops().items() if k not in ("delay", "barrier", "measure")}.values()))

In [ ]:
to_run[0].depth()

In [ ]:
fig, ax = plt.subplots(figsize=(30, 20));

timeline_drawer(to_run[0], style=IQXSimple(), idle_wires=False, target=computer.target, axis=ax)

In [ ]:
# to_run[0].draw(fold=-1, idle_wires=False)

## Run on exact simulator

In [ ]:
# to_run[0].draw(fold=-1)

In [ ]:
max_mps_bond: int = 16
simulator = AerSimulator(
    method="matrix_product_state",
    enable_truncation=True,
    zero_threshold=None,
    validation_threshold=None,
    matrix_product_state_max_bond_dimension=max_mps_bond,
    matrix_product_state_truncation_threshold=1e-5,
    chop_threshold=None,
)
result = simulator.run(to_run, shots=10_000)
counts_mps = result.result().get_counts()
counts_mps

In [ ]:
pickle.dump(counts_mps, open(fname + f"_mps_chi_{max_mps_bond}_counts", "wb"))

In [ ]:
classical = {}

In [ ]:
for bitstring, count in counts_mps.items():
    print(bitstring.split(" ")[0].count("1"))
    classical[bitstring.split(" ")[0]] = count

In [ ]:
classical

In [ ]:
for b, c in classical.items():
    print(b[:len(cbit)] == cbit[::-1])

In [ ]:
cbit

In [ ]:
stitched = {cbit[::-1] + b: count for b, count in classical_small.items()}
stitched

In [ ]:
for b, c in stitched.items():
    print(b, c)
    print(len(b))
    print(b.count("1"))
    print(b in classical.keys())

In [ ]:
classical_small

In [ ]:
cbit.count("1")

In [ ]:
bitstring = list(classical.keys())[0]
bitstring

In [ ]:
def _stitch_bitstring(bitstring: str, indices_to_bools: dict[int, str]) -> str:
    new_bitstring = ""
    j = 0
    for i in range(nqubits_original):
        if i in indices_to_bools.keys():
            new_bitstring += indices_to_bools[i]
        else:
            new_bitstring += bitstring[j]
            j += 1
    return new_bitstring


def stitch_samples(counts: dict[str, int], indices_to_bools: dict[int, str]) -> dict[str, int]:
    return {_stitch_bitstring(bitstring, indices_to_bools): count for bitstring, count in counts.items()}

In [ ]:
classical = stitch_samples(classical, indices_to_bools)
classical

In [ ]:
for bitstring, count in classical.items():
    print(bitstring.count("1"))

In [ ]:
# import qiskit.visualization


# qiskit.visualization.plot_histogram(
#     counts,
#     # target_string=hartree_fock_bitstring,
#     # sort="hamming",
#     number_to_keep=50,
#     figsize=(10, 15),
#     title=computer.name,
# )

In [ ]:
# hartree_fock_bitstring = list(counts.keys())[0]
# hartree_fock_bitstring

## Run on noisy simulator

In [ ]:
# sim = AerSimulator.from_backend(computer, method="matrix_product_state")

In [ ]:
# result = sim.run(to_run, shots=nshots)
# counts = result.result().get_counts()

In [ ]:
# qiskit.visualization.plot_histogram(
#     counts,
#     target_string=hartree_fock_bitstring,
#     sort="hamming",
#     number_to_keep=10,
#     figsize=(7, 8),
#     title=sim.name,
# )

## Run on hardware

In [ ]:
nshots: int = 500_000

In [ ]:
job = sampler.run(to_run, shots=nshots)
# job = service.job("d6u2l6atnsts73ermmq0")

In [ ]:
all_counts_hardware = []

In [ ]:
res = job.result()

In [ ]:
res[0].data

In [ ]:
# counts = pickle.load(open("./results/truncated_pool/atp_0_be2_f18/boston/dd_and_postselectprep/atp_0_be2_f18_005_adaptiterations.qasm_counts_ibm_boston_nshots_500000_2026_03_18_22-45-38_dd_postselectprep", "rb"))
# all_counts_hardware = [counts]

In [ ]:
nqubits = res[0].data.meas.num_bits

In [ ]:
# Post select all zero bitstrings on initial measurement.
if use_initial_measurement:
    r = res[0]
    imeas = r.data.meas

    imask = []
    for i, row in enumerate(imeas.to_bool_array()):
        if list(row) == [False] * nqubits:
            imask.append(i)
    print("Post-selection rate:", len(imask) / nshots)

    assert len(imask) == imeas.get_counts().get("0" * nqubits)
    assert not np.any(imeas[imask].to_bool_array())

In [ ]:
for r in res:
    if use_initial_measurement:
        all_counts_hardware.append(r.data.meas0[imask].get_counts())
    else:
        all_counts_hardware.append(r.data.meas.get_counts())

In [ ]:
len(all_counts_hardware[0])

In [ ]:
hamming_weight = num_electrons
num_correct_hamming_weight = 0
hamming_weights = []
for bitstring, count in all_counts_hardware[0].items():
    hamming_weights.append(bitstring.count("1"))
    if bitstring.count("1") == hamming_weight:
        num_correct_hamming_weight += count
num_correct_hamming_weight / sum(all_counts_hardware[0].values())

In [ ]:
counts = np.bincount(hamming_weights)
fractions = counts / counts.sum()
x = np.arange(len(counts))
plt.figure(figsize=(12, 5))
plt.bar(x, fractions, width=0.8, align='center')
plt.xlabel("Fermion Number")
plt.ylabel("Fraction")

min_hw = min(hamming_weights)
max_hw = max(hamming_weights)
plt.xlim(min_hw - 0.5, max_hw + 0.5)
plt.xticks(np.arange(min_hw, max_hw + 1))

plt.title(fragment + f"_counts_{computer.name}_nshots_{nshots}" + "_dd" * use_dd + "_postselectprep" * use_initial_measurement + "_lucj")
plt.axvline(hamming_weight, label="Correct Fermion Number", color="xkcd:green")
plt.legend()
plt.show();

In [ ]:
def is_valid_bitstring(
    bitstring: str, norb: int, nelec: tuple[int, int]
) -> bool:
    n_alpha, n_beta = nelec
    return (
        len(bitstring) == 2 * norb
        and bitstring[norb:].count("1") == n_alpha
        and bitstring[:norb].count("1") == n_beta
    )


bit_array = res[0].data.meas0
num_valid = sum(
    is_valid_bitstring(b, norb, nelec) for b in bit_array.get_bitstrings()
)
valid_fraction = num_valid / bit_array.num_shots
print(f"Fraction of sampled configurations that are valid: {valid_fraction}")

In [ ]:
import math

expected_fraction_random = (
    math.comb(norb, n_alpha) * math.comb(norb, n_beta) / 2 ** (2 * norb)
)
print(
    f"Expected fraction of valid configurations from uniformly random bitstrings: {expected_fraction_random}"
)

In [ ]:
import qiskit.visualization


qiskit.visualization.plot_histogram(
    all_counts_hardware[0],
    # target_string=hartree_fock_bitstring,
    # sort="hamming",
    number_to_keep=50,
    figsize=(10, 15),
    title=computer.name,
)

## Save data

In [ ]:
import datetime


time_key = datetime.datetime.now().strftime("%Y_%m_%d_%H-%M-%S")
fname = fragment + f"_{adapt_iterations:03d}_lucjiterations" + f"_{computer.name}_nshots_{nshots}_{time_key}" + "_dd" * use_dd + "_postselectprep" * use_initial_measurement
fname

In [ ]:
pickle.dump(all_counts_hardware[0], open(fname, "wb"))

## Run SQD

In [ ]:
import pyscf.tools
from pyscf import ao2mo

import collections
from functools import partial
import os
import pickle

from qiskit.primitives import BitArray
from qiskit_addon_sqd.fermion import SCIResult, diagonalize_fermionic_hamiltonian, solve_sci_batch

In [ ]:
mode_order = pickle.load(
    open(f"{circuit_dir}/{fragment}/{fragment}_mode_order_{adapt_iterations:03d}_adaptiterations.pkl", "rb")
)
qubit_order = pickle.load(
    open(f"{circuit_dir}/{fragment}/{fragment}_qubit_order_{adapt_iterations:03d}_adaptiterations.pkl", "rb")
)

In [ ]:
hamiltonian_dir = "hamiltonians"

In [ ]:
fcidump = pyscf.tools.fcidump.read(f"{hamiltonian_dir}/{fragment}.fcidump")

In [ ]:
n_orbitals = fcidump.get("NORB")
num_electrons = fcidump.get("NELEC")
ecore = fcidump.get("ECORE")
h1 = fcidump.get("H1")
h2 = fcidump.get("H2")
h2 = ao2mo.restore(1, h2, n_orbitals)

In [ ]:
import pyscf.cc
import pyscf.ao2mo
import numpy as np
import ffsim

# --- 1. Parse your fcidump (you already have this) ---
norb = fcidump.get("NORB")
num_electrons = fcidump.get("NELEC")
ms2 = fcidump.get("MS2", 0)
ecore = fcidump.get("ECORE")
h1e = fcidump.get("H1")
h2e = fcidump.get("H2")
h2e = pyscf.ao2mo.restore(1, h2e, norb)  # restore to full 4-index (norb,norb,norb,norb)

n_alpha = (num_electrons + ms2) // 2
n_beta  = (num_electrons - ms2) // 2
nelec = (n_alpha, n_beta)

# --- 2. Run CCSD directly on the fcidump integrals ---
# Build a fake RHF object whose integrals come from the fcidump
mol = pyscf.gto.M()   # empty Mole — no basis set needed
mol.nelectron = num_electrons
mol.spin = ms2

# Construct a minimal RHF object and inject the integrals manually
mf = pyscf.scf.RHF(mol)
mf.get_hcore = lambda *args: h1e
mf.get_ovlp  = lambda *args: np.eye(norb)
mf._eri = pyscf.ao2mo.restore(8, h2e, norb)  # 8-fold symmetry for RHF
mf.kernel()  # actually solve HF — gives proper mo_coeff, mo_occ, and e_tot

# Run CCSD on top of the converged HF
ccsd = pyscf.cc.RCCSD(mf)
ccsd.kernel()
t1, t2 = ccsd.t1, ccsd.t2

In [ ]:
mol_data = ffsim.MolecularData(
    norb=norb,
    nelec=nelec,
    core_energy=ecore,
    one_body_integrals=h1e,
    two_body_integrals=h2e,
    hf_energy=mf.e_tot,
    ccsd_energy=ccsd.e_tot,
    ccsd_t1=t1,
    ccsd_t2=t2,
)

In [ ]:
# pairs_aa = [(p, p + 1) for p in range(norb - 1)]
# pairs_ab = [(p, p)     for p in range(norb)]
# pairs_aa = [(p, p + 1) for p in range(norb - 1)]
# pairs_ab = [(p, p)     for p in range(0, norb, 4)]
pairs_aa = [(p, p + 1) for p in range(norb - 1)]
pairs_ab = [(p, p)     for p in range(0, norb, 4)]
n_reps = 1

lucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=mol_data.ccsd_t2,
    t1=mol_data.ccsd_t1,
    n_reps=n_reps,
    interaction_pairs=(pairs_aa, pairs_ab),
    optimize=True,
    options=dict(maxiter=50),
    regularization=1e-2,
)

In [ ]:
qubits = qiskit.QuantumRegister(2 * norb, name="q")
circuit = qiskit.QuantumCircuit(qubits)
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), qubits)
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(lucj_op), qubits)

In [ ]:
from typing import Sequence

import rustworkx
from qiskit.providers import BackendV2
from rustworkx import NoEdgeBetweenNodes, PyGraph

IBM_TWO_Q_GATES = {"cx", "ecr", "cz"}


def create_linear_chains(num_orbitals: int) -> PyGraph:
    """In zig-zag layout, there are two linear chains (with connecting qubits between
    the chains). This function creates those two linear chains: a rustworkx PyGraph
    with two disconnected linear chains. Each chain contains `num_orbitals` number
    of nodes, that is, in the final graph there are `2 * num_orbitals` number of nodes.

    Args:
        num_orbitals (int): Number orbitals or nodes in each linear chain. They are
            also known as alpha-alpha interaction qubits.

    Returns:
        A rustworkx.PyGraph with two disconnected linear chains each with `num_orbitals`
            number of nodes.
    """
    G = rustworkx.PyGraph()

    for n in range(num_orbitals):
        G.add_node(n)

    for n in range(num_orbitals - 1):
        G.add_edge(n, n + 1, None)

    for n in range(num_orbitals, 2 * num_orbitals):
        G.add_node(n)

    for n in range(num_orbitals, 2 * num_orbitals - 1):
        G.add_edge(n, n + 1, None)

    return G


def create_lucj_zigzag_layout(
    num_orbitals: int, backend_coupling_graph: PyGraph
) -> tuple[PyGraph, int]:
    """This function creates the complete zigzag graph that 'can be mapped' to an IBM QPU with
    heavy-hex connectivity (the zigzag must be an isomorphic sub-graph to the QPU/backend
    coupling graph for it to be mapped).
    The zigzag pattern includes both linear chains (alpha-alpha interactions) and connecting
    qubits between the linear chains (alpha-beta interactions).

    Args:
        num_orbitals (int): Number of orbitals, that is, number of nodes in each alpha-alpha linear chain.
        backend_coupling_graph (PyGraph): The coupling graph of the backend on which the LUCJ ansatz
            will be mapped and run. This function takes the coupling graph as a undirected
            `rustworkx.PyGraph` where there is only one 'undirected' edge between two nodes,
            that is, qubits. Usually, the coupling graph of a IBM backend is directed (for example, Eagle devices
            such as ibm_brisbane) or may have two edges between two nodes (for example, Heron `ibm_torino`).
            A user needs to be make such graphs undirected and/or remove duplicate edges to make them
            compatible with this function.

    Returns:
        G_new (PyGraph): The graph with IBM backend compliant zigzag pattern.
        num_alpha_beta_qubits (int): Number of connecting qubits between the linear chains
            in the zigzag pattern. While we want as many connecting (alpha-beta) qubits between
            the linear (alpha-alpha) chains, we cannot accommodate all due to qubit and connectivity
            constraints of backends. This is the maximum number of connecting qubits the zigzag pattern
            can have while being backend compliant (that is, isomorphic to backend coupling graph).
    """
    isomorphic = False
    G = create_linear_chains(num_orbitals=num_orbitals)

    num_iters = num_orbitals
    while not isomorphic:
        G_new = G.copy()
        num_alpha_beta_qubits = 0
        for n in range(num_iters):
            if n % 4 == 0:
                new_node = 2 * num_orbitals + num_alpha_beta_qubits
                G_new.add_node(new_node)
                G_new.add_edge(n, new_node, None)
                G_new.add_edge(new_node, n + num_orbitals, None)
                num_alpha_beta_qubits = num_alpha_beta_qubits + 1
        isomorphic = rustworkx.is_subgraph_isomorphic(
            backend_coupling_graph, G_new
        )
        num_iters -= 1

    return G_new, num_alpha_beta_qubits


def lightweight_layout_error_scoring(
    backend: BackendV2,
    virtual_edges: Sequence[Sequence[int]],
    physical_layouts: Sequence[int],
    two_q_gate_name: str,
) -> list[list[list[int], float]]:
    """Lightweight and heuristic function to score isomorphic layouts. There can be many zigzag patterns,
    each with different set of physical qubits, that can be mapped to a backend. Some of them may
    include less noise qubits and couplings than others. This function computes a simple error score
    for each such layout. It sums up 2Q gate error for all couplings in the zigzag pattern (layout) and
    measurement of errors of physical qubits in the layout to compute the error score.

    Note:
        This lightweight scoring can be refined using concepts such as mapomatic.

    Args:
        backend (BackendV2): A backend.
        virtual_edges (Sequence[Sequence[int]]): Edges in the device compliant zigzag pattern where
            nodes are numbered from 0 to (2 * num_orbitals + num_alpha_beta_qubits).
        physical_layouts (Sequence[int]): All physical layouts of the zigzag pattern that are isomorphic
            to each other and to the larger backend coupling map.
        two_q_gate_name (str): The name of the two-qubit gate of the backend. The name is used for fetching
            two-qubit gate error from backend properties.

    Returns:
        scores (list): A list of lists where each sublist contains two items. First item is the layout, and
            second item is a float representing error score of the layout. The layouts in the `scores` are
            sorted in the ascending order of error score.
    """
    props = backend.properties()
    scores = []
    for layout in physical_layouts:
        total_2q_error = 0
        for edge in virtual_edges:
            physical_edge = (layout[edge[0]], layout[edge[1]])
            try:
                ge = props.gate_error(two_q_gate_name, physical_edge)
            except Exception:
                ge = props.gate_error(two_q_gate_name, physical_edge[::-1])
            total_2q_error += ge
        total_measurement_error = 0
        for qubit in layout:
            meas_error = props.readout_error(qubit)
            total_measurement_error += meas_error
        scores.append([layout, total_2q_error + total_measurement_error])

    return sorted(scores, key=lambda x: x[1])


def _make_backend_cmap_pygraph(backend: BackendV2) -> PyGraph:
    graph = backend.coupling_map.graph
    if not graph.is_symmetric():
        graph.make_symmetric()
    backend_coupling_graph = graph.to_undirected()

    edge_list = backend_coupling_graph.edge_list()
    removed_edge = []
    for edge in edge_list:
        if set(edge) in removed_edge:
            continue
        try:
            backend_coupling_graph.remove_edge(edge[0], edge[1])
            removed_edge.append(set(edge))
        except NoEdgeBetweenNodes:
            pass

    return backend_coupling_graph


def get_zigzag_physical_layout(
    num_orbitals: int, backend: BackendV2, score_layouts: bool = True
) -> tuple[list[int], int]:
    """The main function that generates the zigzag pattern with physical qubits that can be used
    as an `intial_layout` in a preset passmanager/transpiler.

    Args:
        num_orbitals (int): Number of orbitals.
        backend (BackendV2): A backend.
        score_layouts (bool): Optional. If `True`, it uses the `lightweight_layout_error_scoring`
            function to score the isomorphic layouts and returns the layout with less erroneous qubits.
            If `False`, returns the first isomorphic subgraph.

    Returns:
        A tuple of device compliant layout (list[int]) with zigzag pattern and an int representing
            number of alpha-beta-interactions.
    """
    backend_coupling_graph = _make_backend_cmap_pygraph(backend=backend)

    G, num_alpha_beta_qubits = create_lucj_zigzag_layout(
        num_orbitals=num_orbitals,
        backend_coupling_graph=backend_coupling_graph,
    )

    isomorphic_mappings = rustworkx.vf2_mapping(
        backend_coupling_graph, G, subgraph=True
    )
    isomorphic_mappings = list(isomorphic_mappings)

    edges = list(G.edge_list())

    layouts = []
    for mapping in isomorphic_mappings:
        initial_layout = [None] * (2 * num_orbitals + num_alpha_beta_qubits)
        for key, value in mapping.items():
            initial_layout[value] = key
        layouts.append(initial_layout)

    two_q_gate_name = IBM_TWO_Q_GATES.intersection(
        backend.configuration().basis_gates
    ).pop()

    if score_layouts:
        scores = lightweight_layout_error_scoring(
            backend=backend,
            virtual_edges=edges,
            physical_layouts=layouts,
            two_q_gate_name=two_q_gate_name,
        )

        return scores[0][0][:-num_alpha_beta_qubits], num_alpha_beta_qubits

    return layouts[0][:-num_alpha_beta_qubits], num_alpha_beta_qubits

In [ ]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

In [ ]:
initial_layout, _ = get_zigzag_physical_layout(norb, backend=computer)

In [ ]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=computer, initial_layout=initial_layout
)

# without PRE_INIT passes
isa_circuit = pass_manager.run(circuit)
print(f"Gate counts (w/o pre-init passes): {isa_circuit.count_ops()}")

# with PRE_INIT passes
# We will use the circuit generated by this pass manager for hardware execution
pass_manager.pre_init = ffsim.qiskit.PRE_INIT
isa_circuit = pass_manager.run(circuit)
print(f"Gate counts (w/ pre-init passes): {isa_circuit.count_ops()}")

In [ ]:
circuit = isa_circuit.copy()

In [ ]:
circuit.count_ops().get("cz")

In [ ]:
from qiskit.visualization import timeline_drawer
from qiskit.visualization.timeline.stylesheet import IQXStandard, IQXSimple, IQXDebugging

fig, ax = plt.subplots(figsize=(30, 20));

timeline_drawer(circuit, style=IQXSimple(), idle_wires=False, target=computer.target, axis=ax)

In [ ]:
from pyscf import gto, scf, cc


mol = gto.M()
mol.nelectron = num_electrons
mol.spin = 0

mf = scf.RHF(mol)
mf.get_hcore = lambda *args: h1
mf.get_ovlp = lambda *args: np.eye(n_orbitals)
mf._eri = h2

# HF
mf.kernel()

# CCSD
mycc = cc.CCSD(mf)
mycc.kernel()

# CCSD(T)
et = mycc.ccsd_t()

print("HF energy:   ", mf.e_tot + ecore)
print("CCSD energy: ", mycc.e_tot + ecore)
print("CCSD(T) energy: ", mycc.e_tot + et + ecore)

In [ ]:
# from pyscf import fci

# cisolver = fci.FCI(mf)
# cisolver.verbose = 4

# e_fci, fcivec = cisolver.kernel(h1, h2, n_orbitals, num_electrons)

# print(f"FCI energy: {e_fci + ecore:.12f}")

In [ ]:
# from pyscf import fci

# # Create selected CI solver
# cisolver = fci.SCI(mf)

# # Optional: control accuracy vs cost
# cisolver.select_cutoff = 1e-3   # looser = faster, tighter = more accurate
# cisolver.ci_coeff_cutoff = 1e-3
# cisolver.verbose = 4

# # Run SCI
# nelec = (num_electrons // 2, num_electrons // 2)  # (18, 18)

# e_sci, fcivec = cisolver.kernel(h1, h2, n_orbitals, nelec)

# print(f"SCI energy: {e_sci + ecore:.12f}")

In [ ]:
# import os
# import tempfile
# import numpy as np
# from pyscf import gto, ao2mo
# from pyscf.tools import fcidump as fcidump_reader
# from pyscf.dmrgscf import DMRGCI

# # Load FCIDUMP

# # Build mol from FCIDUMP metadata
# mol = gto.Mole()
# mol.nelectron = num_electrons
# mol.spin      = fcidump.get('MS2', 0)
# mol.verbose   = 4
# mol.build()

# # Scratch directory
# scratch = tempfile.mkdtemp()
# os.makedirs(scratch, exist_ok=True)

# # Set up DMRG
# dmrgci = DMRGCI(mol, n_orbitals, num_electrons)
# dmrgci.executable       = '/Users/ryan/prof/work/wellcome/CheMPS2/build/CheMPS2/chemps2'
# dmrgci.mpiprefix        = ''
# dmrgci.runtimeDir       = scratch
# dmrgci.scratchDirectory = scratch
# dmrgci.maxM             = 500
# dmrgci.conv_tol         = 1e-6
# dmrgci.verbose          = 4

# # Run
# nelec = (num_electrons // 2, num_electrons // 2)
# e_dmrg, _ = dmrgci.kernel(h1, h2, n_orbitals, nelec)
# print(f"DMRG total energy: {e_dmrg + ecore:.12f}")

In [ ]:
num_electrons

In [ ]:
energy_tol = 1e-8
occupancies_tol = 1e-8
carryover_threshold = 1e-5

In [ ]:
fname = "atp_0_be2_f2_010_adaptiterations.qasm_counts_ibm_fez_nshots_100000_2026_02_24_00-01-41"

In [ ]:
counts = pickle.load(
    open(f"results/{fragment}/{fname}", "rb")
)

In [ ]:
counts = collections.Counter(all_counts_hardware[0])
for cb, cc in classical.items():
    counts[cb] += cc

In [ ]:
assert sum(counts.values()) == sum(all_counts_hardware[0].values()) + sum(classical.values())

In [ ]:
def transform_bitstring(bits):
    """
    Convert a given bitstring from Openfermion convention 
    (alternating alpha/beta, big endian) to Qiskit (all alpha
    then all beta, little endian).
    """

    left = [bits[i] for i in range(len(bits)) if i % 2 == 1]   # beta
    right = [bits[i] for i in range(len(bits)) if i % 2 == 0]  # alpha

    # Reverse each half
    left.reverse()
    right.reverse()

    # Concatenate
    return ''.join(left + right)

In [ ]:
measurement_outcomes = counts
permuted_outcomes = {}
for original_bitstring in measurement_outcomes.keys():
    qubit_permuted_bitstring = "".join([original_bitstring[qubit_order.index(n)] for n in range(nqubits)])
    mode_permuted_bitstring = "".join([qubit_permuted_bitstring[mode_order.index(n)] for n in range(nqubits)])

    final_permuted_bitstring = transform_bitstring(mode_permuted_bitstring)
    permuted_outcomes[final_permuted_bitstring[::]] = measurement_outcomes[original_bitstring]

bit_array = BitArray.from_counts(permuted_outcomes)
counts = bit_array.get_counts()
max_key = max(counts, key=counts.get)
print(f'Most common bitstring: {max_key} with count {counts[max_key]}')
print(f'Total number of bitstrings: {len(counts)}')
print(f"Total number of samples:", sum(counts.values()))

In [ ]:
sci_solver = partial(solve_sci_batch, spin_sq=0, max_cycle=10000)
result_history = []

def callback(results: list[SCIResult]):
    result_history.append(results)
    iteration = len(result_history)
    print(f"Iteration {iteration}")
    for i, result in enumerate(results):
        print(f"\tSubsample {i}")
        print(f"\t\tEnergy: {result.energy + ecore}")
        print(f"\t\tSubspace dimension: {np.prod(result.sci_state.amplitudes.shape)}")


result = diagonalize_fermionic_hamiltonian(
    one_body_tensor=h1,
    two_body_tensor=h2,
    bit_array=bit_array,
    samples_per_batch=500,
    norb=n_orbitals,
    nelec=(num_electrons // 2, num_electrons // 2),
    num_batches=2,
    energy_tol=energy_tol,
    occupancies_tol=occupancies_tol,
    max_iterations=100,
    sci_solver=sci_solver,
    symmetrize_spin=True,
    carryover_threshold=carryover_threshold,
    callback=callback,
)